# 03 · Analysis and visualisation
**Question:** How much of the October–November PM2.5 rise in Delhi comes from Diwali night, and how much from the season's weather?

Input: `clean/delhi_pm25_hourly.parquet` (11 sites × 3 years × hourly, 15 Sep–30 Nov).

**Plan**
1. Build a city-wide PM2.5 series (average of the 11 sites).
2. Measure the seasonal rise: late-September baseline vs October–November.
3. Ask how much of that rise the weather can explain, using a simple regression of PM2.5 on weather.
4. Measure Diwali night against (a) the nights around it and (b) what the weather alone predicts for that night.
5. Turn the Diwali-night excess into its share of the October–November rise.

Charts are built with Plotly (interactive), shown here and also saved as JSON for the website in `docs/`.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
pio.renderers.default = "plotly_mimetype+notebook_connected"   # charts show in Jupyter/VS Code/Colab and in HTML exports
import statsmodels.formula.api as smf

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN, INTERIM, DOCS = ROOT / "clean", ROOT / "interim", ROOT / "docs"
FIG_DIR = DOCS / "assets" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

DIWALI = {2023: pd.Timestamp("2023-11-12"), 2024: pd.Timestamp("2024-10-31"), 2025: pd.Timestamp("2025-10-20")}
YEARS = [2023, 2024, 2025]

# ---- Colours (checked with a colour-blind validator) -------------------------
# Years: one blue, light -> dark, plus a different line style each, so they stay apart in greyscale.
YEAR_COLOR = {2023: "#86b6ef", 2024: "#2a78d6", 2025: "#104281"}
YEAR_DASH = {2023: "dot", 2024: "dash", 2025: "solid"}
HIGHLIGHT = "#eb6834"   # the ONE highlight colour: Diwali
CONTEXT = "#b8b7b0"     # neutral grey for background points
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e7e6e2"

def style(fig, title, xtitle, ytitle, height=430):
    fig.update_layout(
        title=dict(text=title, x=0, xanchor="left", font=dict(size=16, color=INK)),
        template="simple_white", height=height, margin=dict(l=60, r=20, t=70, b=55),
        font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color=INK2),
        legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="right", x=1, title=None),
        hoverlabel=dict(bgcolor="white", font_size=12), plot_bgcolor="white", paper_bgcolor="white")
    fig.update_xaxes(title=xtitle, showgrid=False, linecolor=GRID, ticks="outside", tickcolor=GRID)
    fig.update_yaxes(title=ytitle, showgrid=True, gridcolor=GRID, linecolor=GRID, zeroline=False)
    return fig

def save(fig, name):
    """Save the chart for the website and show it here."""
    (FIG_DIR / f"{name}.json").write_text(fig.to_json(), encoding="utf-8")
    fig.show()

RESULTS = {}

## 1 · City-wide PM2.5
For each hour we average the 11 sites, but only if at least 8 of them report (so one station's gap cannot swing the city value). Daily means need 18 of 24 hours; "night" means 18:00 to 05:59 and needs 9 of 12 hours.

In [2]:
d = pd.read_parquet(CLEAN / "delhi_pm25_hourly.parquet")
d["station_key"] = d["station_key"].astype(str)
n_sites = d.groupby("ts_ist")["pm25"].count()
city = (d.groupby("ts_ist")["pm25"].mean().where(n_sites >= 8).rename("pm25").to_frame()
          .join(d.drop_duplicates("ts_ist").set_index("ts_ist")[["year", "days_from_diwali", "temp_c", "rh_pct", "rain_mm", "wind_kmh", "blh_m"]]))
print("City hours:", len(city), "| hours without a city value:", city.pm25.isna().sum())

daily = (city.groupby(city.index.normalize())
             .agg(pm25=("pm25", "mean"), n=("pm25", "count"), blh=("blh_m", "mean"), wind=("wind_kmh", "mean"),
                  temp=("temp_c", "mean"), rh=("rh_pct", "mean"), rain=("rain_mm", "sum"), year=("year", "first")))
daily.loc[daily.n < 18, "pm25"] = np.nan
daily["dd"] = [(i - DIWALI[y]).days for i, y in zip(daily.index, daily.year)]
daily["month"] = daily.index.month

city["night_of"] = (city.index - pd.Timedelta(hours=18)).normalize()
nights = (city[(city.index.hour >= 18) | (city.index.hour < 6)].groupby("night_of")
              .agg(pm25=("pm25", "mean"), n=("pm25", "count"), blh=("blh_m", "mean"), wind=("wind_kmh", "mean"),
                   temp=("temp_c", "mean"), rh=("rh_pct", "mean"), year=("year", "first")))
nights.loc[nights.n < 9, "pm25"] = np.nan
nights["dd"] = [(i - DIWALI[y]).days for i, y in zip(nights.index, nights.year)]
nights = nights[nights.index.month * 100 + nights.index.day <= 1129]   # the 30 Nov night runs past the window
print("Days:", len(daily), "| nights:", len(nights))

City hours: 5544 | hours without a city value: 0
Days: 231 | nights: 231


## 2 · Summary statistics and the seasonal rise

In [3]:
def period(m):
    return np.where(m == 9, "15-30 Sep", "Oct-Nov")
city["period"] = period(city.index.month)
summary = (city.groupby(["year", "period"])["pm25"].agg(["mean", "median", "max", "count"]).round(0).unstack())
rise = (city[city.period == "Oct-Nov"].groupby("year").pm25.mean() - city[city.period == "15-30 Sep"].groupby("year").pm25.mean()).round(1)
summary[("rise", "Oct-Nov minus Sep")] = rise
summary

mean            median               max             count  \
period 15-30 Sep Oct-Nov 15-30 Sep Oct-Nov 15-30 Sep Oct-Nov 15-30 Sep   
year                                                                     
2023        47.0   180.0      48.0   160.0      99.0   679.0       384   
2024        52.0   184.0      47.0   167.0     139.0   857.0       384   
2025        47.0   174.0      45.0   151.0     113.0  1331.0       384   

                            rise  
period Oct-Nov Oct-Nov minus Sep  
year                              
2023      1464             132.9  
2024      1464             131.7  
2025      1464             126.3

In [4]:
RESULTS["sep_mean"] = city[city.period == "15-30 Sep"].groupby("year").pm25.mean().round(1).to_dict()
RESULTS["octnov_mean"] = city[city.period == "Oct-Nov"].groupby("year").pm25.mean().round(1).to_dict()
RESULTS["rise"] = rise.to_dict()
NAAQS_24H = 60   # India's 24-hour standard for PM2.5 (µg/m³)
over = daily[daily.month >= 10].groupby("year").apply(lambda x: (x.pm25 > NAAQS_24H).sum() / x.pm25.notna().sum() * 100).round(0)
RESULTS["pct_days_over_naaqs_octnov"] = over.to_dict()
print("Share of Oct-Nov days above India's 24-h standard (60):", over.to_dict())

# By station: Oct-Nov mean
st = d[d.ts_ist.dt.month >= 10].groupby(["station_key", "year"]).pm25.mean().unstack().round(0)
st["mean"] = st.mean(axis=1).round(0)
st.sort_values("mean", ascending=False)

Share of Oct-Nov days above India's 24-h standard (60): {2023: 95.0, 2024: 93.0, 2025: 89.0}


/sessions/rcw-01a8zy71zkermwkg9fl4twy5/tmp/ipykernel_7/1795557951.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  over = daily[daily.month >= 10].groupby("year").apply(lambda x: (x.pm25 > NAAQS_24H).sum() / x.pm25.notna().sum() * 100).round(0)


year,2023,2024,2025,mean
station_key,,,,
jahangirpuri,216.0,222.0,210.0,216.0
anand_vihar,209.0,232.0,198.0,213.0
punjabi_bagh,207.0,206.0,189.0,201.0
rk_puram,195.0,190.0,186.0,190.0
narela,183.0,192.0,188.0,188.0
dwarka_sec8,188.0,189.0,173.0,183.0
okhla_ph2,179.0,175.0,163.0,172.0
ito,168.0,163.0,176.0,169.0
najafgarh,158.0,164.0,141.0,154.0


### Chart 1 (univariate): every day of the season as a dot
A box plot would hide the shape of each month (lecture 6), so we draw every day as a dot, jittered sideways, with a black tick at the median.

In [5]:
rng = np.random.default_rng(0)
MONTH_POS = {9: 0, 10: 1, 11: 2}
YEAR_OFF = {2023: -0.26, 2024: 0.0, 2025: 0.26}
dd1 = daily[daily.pm25.notna()].copy()
fig1 = go.Figure()
for y in YEARS:
    sub = dd1[dd1.year == y]
    xpos = sub.month.map(MONTH_POS) + YEAR_OFF[y] + rng.uniform(-0.08, 0.08, len(sub))
    fig1.add_trace(go.Scatter(x=xpos, y=sub.pm25, mode="markers", name=str(y),
                              marker=dict(color=YEAR_COLOR[y], size=7, opacity=0.85, line=dict(color="white", width=0.5),
                                          symbol={2023: "circle-open", 2024: "diamond", 2025: "circle"}[y]),
                              customdata=sub.index.strftime("%d %b %Y"), hovertemplate="%{customdata}<br>%{y:.0f} µg/m³<extra></extra>"))
med = dd1.groupby(["month", "year"]).pm25.median()
fig1.add_trace(go.Scatter(x=[MONTH_POS[m] + YEAR_OFF[y] for m, y in med.index], y=med.values, mode="markers", name="Median",
                          marker=dict(symbol="line-ew", size=26, line=dict(color=INK, width=3)),
                          hovertemplate="median %{y:.0f} µg/m³<extra></extra>"))
fig1.add_hline(y=NAAQS_24H, line=dict(color=INK2, width=1, dash="dot"),
               annotation_text="India 24-h standard (60)", annotation_position="top left", annotation_font_color=INK2)
style(fig1, "Every day of late September sits near the safe limit; almost every November day is several times above it",
      "", "Daily city-wide PM2.5 (µg/m³)")
fig1.update_xaxes(tickvals=[0, 1, 2], ticktext=["September (15-30)", "October", "November"], range=[-0.5, 2.5])
RESULTS["daily_median_by_month"] = med.round(0).unstack().to_dict()
save(fig1, "fig1_monthly_distribution")

### Chart 2: the whole season, one panel per year (small multiples, same axes)

In [6]:
fig2 = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05)
for i, y in enumerate(YEARS, start=1):
    sub = daily[daily.year == y]
    x = pd.to_datetime("2000-" + sub.index.strftime("%m-%d"))
    fig2.add_trace(go.Scatter(x=x, y=sub.pm25, mode="lines", line=dict(color=YEAR_COLOR[2025], width=2), showlegend=False,
                              fill="tozeroy", fillcolor="rgba(16,66,129,0.07)",
                              customdata=sub.index.strftime("%d %b %Y"), hovertemplate="%{customdata}<br>%{y:.0f} µg/m³<extra></extra>"), row=i, col=1)
    dwt = pd.Timestamp(2000, DIWALI[y].month, DIWALI[y].day)
    fig2.add_vrect(x0=dwt - pd.Timedelta(hours=12), x1=dwt + pd.Timedelta(hours=36), fillcolor=HIGHLIGHT, opacity=0.18, line_width=0, row=i, col=1)
    fig2.add_annotation(x=dwt, y=1, yref=f"y{'' if i == 1 else i} domain", text=f"Diwali {DIWALI[y]:%d %b}", showarrow=False,
                        yanchor="bottom", font=dict(color="#c4501f", size=11), row=i, col=1)
    fig2.add_hline(y=NAAQS_24H, line=dict(color=INK2, width=1, dash="dot"), row=i, col=1)
    fig2.add_annotation(x=0, xref="paper", y=1, yref=f"y{'' if i == 1 else i} domain", text=f"<b>{y}</b>", showarrow=False,
                        xanchor="left", yanchor="top", font=dict(size=13, color=INK))
style(fig2, "The same climb every year from mid-October; Diwali (orange) lands at a different point on it each time", None, None, height=560)
fig2.update_yaxes(range=[0, 620], title_text=None)
fig2.update_yaxes(title_text="Daily PM2.5 (µg/m³)", row=2, col=1)
fig2.update_xaxes(tickformat="%d %b", row=3, col=1)
save(fig2, "fig2_season_daily")

## 3 · How much of the seasonal rise does the weather explain?
As the season moves into winter, the air near the ground gets colder and stiller, and the **boundary-layer height** (how deep the mixing layer is) drops, so smoke has less room to spread. First we simply look at how strongly each weather variable moves with daily PM2.5.

In [7]:
rho_d = daily[daily.pm25.notna() & ~daily.dd.isin([0, 1])][["pm25", "blh", "wind", "temp", "rh", "rain"]].corr(method="spearman")["pm25"].drop("pm25")
RESULTS["spearman_daily"] = rho_d.round(2).to_dict()
rho_d.round(2).to_frame("Spearman correlation with daily PM2.5")

,Spearman correlation with daily PM2.5
blh,-0.21
wind,-0.36
temp,-0.74
rh,-0.30
rain,-0.46


### Chart 3 (bivariate): daily PM2.5 against three weather variables

In [8]:
from plotly.subplots import make_subplots
panels = [("blh", "Mixing-layer height (m, log)", True), ("wind", "Wind speed (km/h)", False), ("temp", "Air temperature (°C)", False)]
fig3 = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.04,
                     subplot_titles=[f"{t.split(' (')[0]}: ρ = {rho_d[c]:.2f}" for c, t, _ in panels])
is_dw_day = daily.dd == 0
for i, (c, t, logx) in enumerate(panels, start=1):
    fig3.add_trace(go.Scatter(x=daily.loc[~is_dw_day, c], y=daily.loc[~is_dw_day, "pm25"], mode="markers", name="Other days",
                              marker=dict(color=CONTEXT, size=7, opacity=0.8, line=dict(color="white", width=1)), showlegend=(i == 1),
                              customdata=daily.index[~is_dw_day].strftime("%d %b %Y"),
                              hovertemplate="%{customdata}<br>%{x:.1f}<br>PM2.5 %{y:.0f} µg/m³<extra></extra>"), row=1, col=i)
    fig3.add_trace(go.Scatter(x=daily.loc[is_dw_day, c], y=daily.loc[is_dw_day, "pm25"], mode="markers", name="Diwali day",
                              marker=dict(color=HIGHLIGHT, size=11, line=dict(color="white", width=2)), showlegend=(i == 1),
                              customdata=daily.index[is_dw_day].strftime("%d %b %Y"),
                              hovertemplate="Diwali %{customdata}<br>%{x:.1f}<br>PM2.5 %{y:.0f} µg/m³<extra></extra>"), row=1, col=i)
    fig3.update_xaxes(title_text=t, type="log" if logx else "linear", row=1, col=i)
style(fig3, "Daily PM2.5 tracks temperature closely; mixing height and wind only loosely", None, None, height=470)
fig3.update_yaxes(title_text="Daily city PM2.5 (µg/m³)", row=1, col=1)
for i, (c, t, logx) in enumerate(panels, start=1):
    fig3.update_xaxes(title_text=t, type="log" if logx else "linear", row=1, col=i)
fig3.update_layout(legend=dict(orientation="h", y=-0.28, yanchor="top", x=0, xanchor="left"), margin=dict(b=110))
for a in fig3.layout.annotations:
    a.font = dict(size=13, color=INK)
save(fig3, "fig3_weather_scatter")

**What the correlations say.** Temperature moves closely with PM2.5 (ρ ≈ −0.7), while mixing height and wind move only loosely with it (ρ between −0.2 and −0.35). But temperature is also a clock: it falls steadily from September to November, exactly when crop-residue burning in Punjab and Haryana and winter heating ramp up. We have no data on those. So "temperature" partly stands in for "the time of year", not only for "the weather".

That is why we fit **two** models of daily PM2.5 (using every day except Diwali and the day after) and treat the truth as lying between them:

| Model | Formula | What it counts as "weather" |
|---|---|---|
| **Ventilation only** (lower estimate) | `log(PM2.5) = a + b·log(mixing height) + c·wind` | Only how well the air can carry pollution away |
| **Full weather** (upper estimate) | `… + d·temperature + e·humidity` | Also the cooling of the season, which overlaps with stubble burning |

We use the log of PM2.5 because weather effects are multiplicative (a lid twice as low roughly doubles the concentration at any level). Then we ask each model: *given only the weather, how much higher should October–November be than late September?*

In [9]:
fit_days = daily[~daily.dd.isin([0, 1]) & daily.pm25.notna()].copy()
MODELS = {"ventilation": "np.log(pm25) ~ np.log(blh) + wind",
          "full": "np.log(pm25) ~ np.log(blh) + wind + temp + rh"}
fits, loyo = {}, []
for name, f in MODELS.items():
    m = smf.ols(f, data=fit_days).fit()
    fits[name] = m
    smear = np.exp(m.resid).mean()          # converts a log-scale prediction back to µg/m³ without bias
    daily[f"pred_{name}"] = np.exp(m.predict(daily)) * smear
    for y in YEARS:                          # out-of-sample check: train on two years, predict the third
        mm = smf.ols(f, data=fit_days[fit_days.year != y]).fit()
        t = fit_days[fit_days.year == y]; lp = np.log(t.pm25)
        loyo.append({"model": name, "left_out_year": y, "R2_unseen_year": 1 - ((lp - mm.predict(t)) ** 2).sum() / ((lp - lp.mean()) ** 2).sum()})
print(fits["full"].summary().tables[1])
fit_table = pd.DataFrame({n: {"R2 (all days)": m.rsquared, "days": int(m.nobs)} for n, m in fits.items()}).T
loyo = pd.DataFrame(loyo).pivot(index="model", columns="left_out_year", values="R2_unseen_year")
RESULTS["model_r2"] = {n: round(m.rsquared, 2) for n, m in fits.items()}
RESULTS["model_r2_unseen_year"] = loyo.round(2).to_dict(orient="index")
pd.concat([fit_table, loyo.add_prefix("R2 unseen ")], axis=1).round(2)

                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       8.5742      0.934      9.181      0.000       6.734      10.415
np.log(blh)     0.1464      0.155      0.943      0.346      -0.159       0.452
wind           -0.0672      0.014     -4.975      0.000      -0.094      -0.041
temp           -0.1258      0.009    -13.290      0.000      -0.144      -0.107
rh             -0.0180      0.003     -5.745      0.000      -0.024      -0.012


,R2 (all days),days,R2 unseen 2023,R2 unseen 2024,R2 unseen 2025
ventilation,0.20,225.0,0.15,0.18,-0.23
full,0.63,225.0,0.63,0.48,0.44


In [10]:
dec = []
for y in YEARS:
    x = daily[daily.year == y]
    sep, on = x.month == 9, x.month >= 10
    row = {"year": y, "observed_sep": x[sep].pm25.mean(), "observed_octnov": x[on].pm25.mean()}
    row["observed_rise"] = row["observed_octnov"] - row["observed_sep"]
    for n in MODELS:
        row[f"rise_{n}"] = x[on][f"pred_{n}"].mean() - x[sep][f"pred_{n}"].mean()
        row[f"share_{n}_%"] = row[f"rise_{n}"] / row["observed_rise"] * 100
    dec.append(row)
dec = pd.DataFrame(dec).set_index("year")
RESULTS["weather_rise"] = dec[["rise_ventilation", "rise_full"]].round(1).to_dict(orient="index")
RESULTS["weather_share_pct"] = dec[["share_ventilation_%", "share_full_%"]].round(0).to_dict(orient="index")
dec.round(1)

,observed_sep,observed_octnov,observed_rise,rise_ventilation,share_ventilation_%,rise_full,share_full_%
year,,,,,,,
2023,47.0,179.9,132.9,4.6,3.5,96.7,72.8
2024,51.9,183.6,131.7,22.9,17.4,95.0,72.1
2025,47.5,173.8,126.3,52.2,41.3,154.3,122.2


**Reading the result.** Ventilation alone (mixing height and wind) predicts only a small part of the rise, and does poorly on a year it has not seen. Adding temperature and humidity makes the model predict most or even all of the rise (more than 100% in 2025, which means it over-shoots). The honest summary: **the seasonal rise is driven by the season, and our weather data cannot cleanly split "stiller, colder air" from "more smoke arriving at this time of year"**. What we *can* pin down is the part that is clearly not the season: Diwali night.

## 4 · Diwali night
"Diwali night" = 18:00 on Diwali day to 05:59 the next morning. We compare it with two baselines:
- **(a) Neighbouring nights:** the average of the nights 2–7 days before and 2–7 days after (skipping the night before and after, which can carry early or leftover firework smoke).
- **(b) Weather-expected:** what a model of night-time PM2.5 on night-time weather predicts for that exact night. This accounts for Diwali falling on a still or a breezy night.

In [11]:
fit_n = nights[~nights.dd.isin([0, 1]) & nights.pm25.notna()]
m_night = smf.ols("np.log(pm25) ~ np.log(blh) + wind + temp + rh", data=fit_n).fit()
smear_n = np.exp(m_night.resid).mean()
nights["pred_weather"] = np.exp(m_night.predict(nights)) * smear_n
print(f"Night model R² = {m_night.rsquared:.2f} on {int(m_night.nobs)} nights")

rows = []
for y in YEARS:
    x = nights[nights.year == y]
    obs = x.loc[x.dd == 0, "pm25"].iloc[0]
    neigh = x[x.dd.abs().between(2, 7)].pm25.mean()
    wexp = x.loc[x.dd == 0, "pred_weather"].iloc[0]
    rows.append({"year": y, "diwali_night": obs, "neighbour_nights": neigh, "excess_vs_neighbours": obs - neigh,
                 "weather_expected": wexp, "excess_vs_weather": obs - wexp, "times_weather_expected": obs / wexp,
                 "blh_m": x.loc[x.dd == 0, "blh"].iloc[0], "wind_kmh": x.loc[x.dd == 0, "wind"].iloc[0]})
dw = pd.DataFrame(rows).set_index("year")
RESULTS["diwali"] = dw.round(1).to_dict(orient="index")
dw.round(1)

Night model R² = 0.65 on 222 nights


,diwali_night,neighbour_nights,excess_vs_neighbours,weather_expected,excess_vs_weather,times_weather_expected,blh_m,wind_kmh
year,,,,,,,,
2023,463.1,271.2,191.9,258.3,204.8,1.8,49.6,6.5
2024,382.1,208.6,173.5,110.3,271.8,3.5,114.2,10.1
2025,785.0,141.8,643.2,93.5,691.5,8.4,170.8,3.7


### From "Diwali night is X µg/m³ higher" to "Diwali night's share of the season"
Diwali night is 12 hours long. October–November is 61 days = 1,464 hours. If Diwali night adds an excess of *E* µg/m³ for 12 hours, it raises the October–November average by *E × 12 / 1,464*. Dividing that by the seasonal rise gives Diwali night's share of the rise. We use the more conservative of the two excess estimates (the smaller one) as our main figure and show both.

In [12]:
HOURS_ON = 61 * 24
share = []
for y in YEARS:
    for basis in ["excess_vs_neighbours", "excess_vs_weather"]:
        e = dw.loc[y, basis]
        add = e * 12 / HOURS_ON
        share.append({"year": y, "basis": basis.replace("excess_vs_", "vs "), "excess_ugm3": e,
                      "adds_to_octnov_mean": add, "share_of_rise_%": add / dec.loc[y, "observed_rise"] * 100})
share = pd.DataFrame(share)
main = share.loc[share.groupby("year").excess_ugm3.idxmin()].set_index("year")
RESULTS["diwali_share_of_rise_pct"] = main["share_of_rise_%"].round(1).to_dict()
RESULTS["diwali_share_range_pct"] = share.groupby("year")["share_of_rise_%"].agg(["min", "max"]).round(1).to_dict(orient="index")
share.round(1)

,year,basis,excess_ugm3,adds_to_octnov_mean,share_of_rise_%
0,2023,vs neighbours,191.9,1.6,1.2
1,2023,vs weather,204.8,1.7,1.3
2,2024,vs neighbours,173.5,1.4,1.1
3,2024,vs weather,271.8,2.2,1.7
4,2025,vs neighbours,643.2,5.3,4.2
5,2025,vs weather,691.5,5.7,4.5


### Chart 4: Diwali night, hour by hour

In [13]:
fig4 = go.Figure()
for y in YEARS:
    start = DIWALI[y] - pd.Timedelta(days=3)
    sub = city.loc[start: DIWALI[y] + pd.Timedelta(days=4)]
    hrs = (sub.index - DIWALI[y]) / pd.Timedelta(hours=1)
    fig4.add_trace(go.Scatter(x=hrs, y=sub.pm25, name=str(y), mode="lines", line=dict(color=YEAR_COLOR[y], width=2, dash=YEAR_DASH[y]),
                              customdata=sub.index.strftime("%d %b %Y %H:%M"),
                              hovertemplate="%{customdata}<br>%{y:.0f} µg/m³<extra></extra>"))
fig4.add_vrect(x0=18, x1=30, fillcolor=HIGHLIGHT, opacity=0.12, line_width=0,
               annotation_text="Diwali night (18:00-06:00)", annotation_position="bottom left", annotation_font_color=INK)
style(fig4, "On Diwali night the city's PM2.5 jumps within a few hours, then takes about a day to fall back",
      "Hours from midnight at the start of Diwali day", "Hourly city-wide PM2.5 (µg/m³)")
fig4.update_xaxes(tickvals=list(range(-72, 97, 24)), ticktext=["-3 d", "-2 d", "-1 d", "Diwali day", "+1 d", "+2 d", "+3 d", "+4 d"])
# Direct labels at each year's peak instead of a legend (proximity)
for tr in fig4.data:
    yv = np.asarray(tr.y, dtype=float); xv = np.asarray(tr.x, dtype=float); k = np.nanargmax(np.where((xv > 12) & (xv < 36), yv, np.nan))
    fig4.add_annotation(x=xv[k], y=yv[k], text=f"<b>{tr.name}</b>: {yv[k]:,.0f}", showarrow=True, arrowhead=0, ax=70, ay={"2023": -20, "2024": 22, "2025": 10}[tr.name],
                        font=dict(color=INK, size=12), arrowcolor=tr.line.color)
fig4.update_layout(showlegend=False)
save(fig4, "fig4_diwali_hourly")

### Chart 5: Diwali night against what the weather predicts

In [14]:
fig5 = go.Figure()
xs = [str(y) for y in YEARS]
fig5.add_trace(go.Bar(x=xs, y=dw.neighbour_nights, name="Neighbouring nights (avg)", marker_color=CONTEXT,
                      hovertemplate="%{x}: %{y:.0f} µg/m³<extra>Neighbouring nights</extra>"))
fig5.add_trace(go.Bar(x=xs, y=dw.weather_expected, name="Expected from that night's weather", marker_color=YEAR_COLOR[2024],
                      hovertemplate="%{x}: %{y:.0f} µg/m³<extra>Weather-expected</extra>"))
fig5.add_trace(go.Bar(x=xs, y=dw.diwali_night, name="Actual Diwali night", marker_color=HIGHLIGHT,
                      text=[f"{v:.0f}" for v in dw.diwali_night], textposition="outside", textfont=dict(color=INK),
                      hovertemplate="%{x}: %{y:.0f} µg/m³<extra>Diwali night</extra>"))
fig5.update_traces(marker_line_width=0)
fig5.update_layout(barmode="group", bargap=0.3, bargroupgap=0.08)
style(fig5, "Diwali night was roughly 2 to 8 times what that night's weather would predict", "Year", "Night-time city PM2.5 (µg/m³)")
save(fig5, "fig5_diwali_vs_expected")

### Chart 6: the answer to the question
For each year: the October–November rise over late September, the part the weather model predicts, and the part Diwali night adds to the season average.

In [15]:
fig6 = go.Figure()
fig6.add_trace(go.Bar(y=xs, x=dec.observed_rise, name="Observed rise (Oct-Nov minus late Sep)", orientation="h", marker_color=CONTEXT,
                      hovertemplate="%{y}: +%{x:.0f} µg/m³<extra>Observed rise</extra>"))
fig6.add_trace(go.Bar(y=xs, x=dec.rise_ventilation, name="Predicted by ventilation only (mixing height, wind)", orientation="h",
                      marker_color=YEAR_COLOR[2023], hovertemplate="%{y}: +%{x:.0f} µg/m³<extra>Ventilation-only model</extra>"))
fig6.add_trace(go.Bar(y=xs, x=dec.rise_full, name="Predicted by full weather (+ temperature, humidity)", orientation="h",
                      marker_color=YEAR_COLOR[2025], hovertemplate="%{y}: +%{x:.0f} µg/m³<extra>Full-weather model</extra>"))
fig6.add_trace(go.Bar(y=xs, x=main.adds_to_octnov_mean, name="Added by Diwali night", orientation="h", marker_color=HIGHLIGHT,
                      text=[f"+{v:.1f}" for v in main.adds_to_octnov_mean], textposition="outside", textfont=dict(color=INK),
                      hovertemplate="%{y}: +%{x:.1f} µg/m³<extra>Diwali night</extra>"))
fig6.update_layout(barmode="group", bargap=0.2, bargroupgap=0.05)
style(fig6, "The season, not Diwali night, drives the October-November rise",
      "Increase in Oct-Nov average PM2.5 over late September (µg/m³)", "", height=520)
fig6.update_yaxes(autorange="reversed", showgrid=False)
fig6.update_xaxes(showgrid=True, gridcolor=GRID)
fig6.update_layout(legend=dict(orientation="v", y=-0.2, yanchor="top", x=0, xanchor="left"), margin=dict(b=150))
save(fig6, "fig6_decomposition")

### Chart 7: which neighbourhoods were hit hardest on Diwali night?

In [16]:
d["night_of"] = (d.ts_ist - pd.Timedelta(hours=18)).dt.normalize()
dn = d[(d.ts_ist.dt.hour >= 18) | (d.ts_ist.dt.hour < 6)]
sn = dn.groupby(["station_key", "night_of"]).pm25.agg(["mean", "count"]).reset_index()
sn = sn[sn["count"] >= 9]
sn["year"] = sn.night_of.dt.year
sn["dd"] = [(n - DIWALI[y]).days for n, y in zip(sn.night_of, sn.year)]
st_ex = []
for (k, y), g in sn.groupby(["station_key", "year"]):
    if (g.dd == 0).any():
        st_ex.append({"station": k, "year": y, "excess": g.loc[g.dd == 0, "mean"].iloc[0] - g[g.dd.abs().between(2, 7)]["mean"].mean()})
st_ex = pd.DataFrame(st_ex)
NAMES = {"anand_vihar": "Anand Vihar", "rk_puram": "R K Puram", "okhla_ph2": "Okhla Ph-2", "punjabi_bagh": "Punjabi Bagh",
         "najafgarh": "Najafgarh", "ito": "ITO", "pusa": "Pusa", "lodhi_road_imd": "Lodhi Road", "jahangirpuri": "Jahangirpuri",
         "narela": "Narela", "dwarka_sec8": "Dwarka Sec-8"}
st_ex["name"] = st_ex.station.map(NAMES)
order = st_ex.groupby("name").excess.mean().sort_values().index.tolist()
fig7 = go.Figure()
for y in YEARS:
    s = st_ex[st_ex.year == y].set_index("name").reindex(order)
    fig7.add_trace(go.Scatter(x=s.excess, y=order, mode="markers", name=str(y),
                              marker=dict(color=YEAR_COLOR[y], size=11, symbol={2023: "circle-open", 2024: "diamond", 2025: "circle"}[y],
                                          line=dict(width=2, color=YEAR_COLOR[y])),
                              hovertemplate="%{y}<br>+%{x:.0f} µg/m³<extra>" + str(y) + "</extra>"))
style(fig7, "2025 was the worst Diwali at almost every station; the outer-edge stations Narela and Najafgarh barely rose in 2023-24",
      "Diwali-night PM2.5 minus neighbouring nights (µg/m³)", "", height=470)
fig7.update_xaxes(showgrid=True, gridcolor=GRID); fig7.update_yaxes(showgrid=False)
save(fig7, "fig7_station_excess")
RESULTS["station_excess_range"] = st_ex.groupby("year").excess.agg(["min", "max"]).round(0).to_dict(orient="index")
st_ex.pivot(index="name", columns="year", values="excess").round(0).loc[order[::-1]]

year,2023,2024,2025
name,,,
R K Puram,NaN,178.0,961.0
Okhla Ph-2,NaN,341.0,784.0
Pusa,288.0,265.0,903.0
ITO,421.0,273.0,673.0
Dwarka Sec-8,136.0,163.0,1003.0
Lodhi Road,213.0,291.0,598.0
Anand Vihar,264.0,216.0,511.0
Punjabi Bagh,76.0,117.0,480.0
Jahangirpuri,244.0,62.0,352.0


## 5 · Data-quality charts
### Chart 8: before and after cleaning (one station where cleaning mattered)

In [17]:
ito = d[(d.station_key == "ito") & (d.year == 2025) & (d.ts_ist.between("2025-11-18", "2025-11-30 23:00"))]
fig8 = go.Figure()
fig8.add_trace(go.Scatter(x=ito.ts_ist, y=ito.pm25_raw, name="Before cleaning (raw)", mode="lines",
                          line=dict(color=CONTEXT, width=2), hovertemplate="%{x|%d %b %H:%M}<br>raw %{y:.0f}<extra></extra>"))
fig8.add_trace(go.Scatter(x=ito.ts_ist, y=ito.pm25, name="After cleaning", mode="lines",
                          line=dict(color=YEAR_COLOR[2025], width=2), hovertemplate="%{x|%d %b %H:%M}<br>clean %{y:.0f}<extra></extra>"))
bad = ito[ito.pm25_flag.isin(["impossible_value", "stuck", "isolated_spike"])]
fig8.add_trace(go.Scatter(x=bad.ts_ist, y=bad.pm25_raw, name="Removed (error)", mode="markers",
                          marker=dict(color=HIGHLIGHT, size=8, symbol="x"),
                          customdata=bad.pm25_flag.astype(str), hovertemplate="%{x|%d %b %H:%M}<br>%{y:.0f} - %{customdata}<extra></extra>"))
fig8.add_hline(y=0, line=dict(color=INK2, width=1))
style(fig8, "Cleaning removed impossible readings: PM2.5 of exactly zero at ITO, late November 2025",
      "Date and hour (IST)", "PM2.5 (µg/m³)")
save(fig8, "fig8_before_after")
print("Removed points in this window:", len(bad), bad.pm25_flag.value_counts().to_dict())

Removed points in this window: 4 {'impossible_value': 4, 'citywide_high_kept': 0, 'isolated_spike': 0, 'ok': 0, 'stuck': 0}


In [18]:
before = d.assign(m=d.pm25_raw.isna()).groupby("station_key").m.mean() * 100
after = d.assign(m=d.pm25.isna()).groupby("station_key").m.mean() * 100
qa = pd.DataFrame({"missing before %": before, "missing after %": after}).round(1)
qa.index = qa.index.map(NAMES)
removed = d[d.pm25_flag.isin(["impossible_value", "stuck", "isolated_spike"])].groupby("station_key").size()
qa["errors removed"] = removed.rename(index=NAMES).reindex(qa.index).fillna(0).astype(int)
RESULTS["cleaning_counts"] = d.pm25_flag.value_counts().to_dict()
RESULTS["source_counts"] = d.pm25_source.value_counts().to_dict()
qa.sort_values("missing before %", ascending=False)

,missing before %,missing after %,errors removed
station_key,,,
Anand Vihar,8.3,5.7,13
Lodhi Road,5.4,5.8,38
R K Puram,4.4,3.4,4
Punjabi Bagh,3.6,3.1,14
Najafgarh,3.4,2.7,7
ITO,2.8,2.5,30
Okhla Ph-2,2.1,1.4,0
Jahangirpuri,1.5,1.0,0
Dwarka Sec-8,1.0,0.7,8


### Chart 9: CPCB vs OpenAQ for the same stations (2025)

In [19]:
match = pd.read_csv(INTERIM / "station_match_cpcb_openaq.csv")
fig9 = go.Figure()
mm = match.copy(); mm["name"] = mm.station_key.map(lambda k: NAMES.get(k, k.replace("pusa_dpcc", "Pusa (DPCC)").replace("pusa_iitm", "Pusa (IITM)")))
mm = mm.sort_values("share_within_0.5")
fig9.add_trace(go.Bar(x=mm["share_within_0.5"] * 100, y=mm["name"], orientation="h", marker_color=YEAR_COLOR[2024],
                      text=[f"{v:.0f}%" for v in mm["share_within_0.5"] * 100], textposition="outside", textfont=dict(color=INK),
                      customdata=np.stack([mm.mean_abs_diff, mm.hours_compared], axis=1),
                      hovertemplate="%{y}<br>%{x:.1f}% of hours identical (±0.5)<br>mean difference %{customdata[0]:.2f} µg/m³<br>%{customdata[1]} hours compared<extra></extra>"))
fig9.add_vline(x=80, line=dict(color=HIGHLIGHT, width=2, dash="dash"), annotation_text="match rule: 80%", annotation_position="bottom right", annotation_font_color=INK)
style(fig9, "OpenAQ is a copy of CPCB data: 11 of 12 stations agree hour-for-hour in at least 80% of hours", "Hours where the two sources agree within 0.5 µg/m³ (%)", "", height=470)
fig9.update_xaxes(range=[0, 112], showgrid=True, gridcolor=GRID); fig9.update_yaxes(showgrid=False)
save(fig9, "fig9_source_agreement")

## 6 · More views of the data
These charts look at the same clean data from more angles: the whole season at a glance, the daily cycle, wind direction, the stations on a map, and the quality of the raw data.

In [20]:
SEQ = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]   # one blue, light -> dark
SEQ_SCALE = [[i / (len(SEQ) - 1), c] for i, c in enumerate(SEQ)]
MONTH_COLOR = {9: "#86b6ef", 10: "#2a78d6", 11: "#104281"}
MONTH_DASH = {9: "dot", 10: "dash", 11: "solid"}
MONTH_NAME = {9: "September (15-30)", 10: "October", 11: "November"}

### Chart 10: the whole season at a glance (calendar heatmap)

In [21]:
cal = daily.copy()
cal["md"] = pd.to_datetime("2000-" + cal.index.strftime("%m-%d"))
z = cal.pivot_table(index="year", columns="md", values="pm25")
fig10 = go.Figure(go.Heatmap(z=z.values, x=z.columns, y=[str(y) for y in z.index], colorscale=SEQ_SCALE, zmin=0, zmax=400,
                             colorbar=dict(title="µg/m³", thickness=12), xgap=1, ygap=3,
                             hovertemplate="%{x|%d %b} %{y}<br>%{z:.0f} µg/m³<extra></extra>"))
fig10.add_trace(go.Scatter(x=[pd.Timestamp(2000, DIWALI[y].month, DIWALI[y].day) for y in YEARS], y=[str(y) for y in YEARS],
                           mode="markers", name="Diwali", marker=dict(symbol="square-open", size=14, color=HIGHLIGHT, line=dict(width=3)),
                           hovertemplate="Diwali %{y}<extra></extra>"))
style(fig10, "Every season turns dark from mid-October; the first 'very poor' day came on 13, 17 and 18 October", "Date", "", height=300)
fig10.update_xaxes(tickformat="%d %b"); fig10.update_yaxes(showgrid=False, autorange="reversed")
save(fig10, "fig10_calendar")
first_poor = {y: daily[(daily.year == y) & (daily.pm25 > 120)].index.min().strftime("%d %b") for y in YEARS}
RESULTS["first_day_above_120"] = first_poor
print("First day with daily mean above 120 (AQI 'poor' or worse):", first_poor)

First day with daily mean above 120 (AQI 'poor' or worse): {2023: '13 Oct', 2024: '17 Oct', 2025: '18 Oct'}


### Chart 11: how many days fell in each official air-quality band
India's National AQI bands for 24-hour PM2.5: Good ≤ 30, Satisfactory 31–60, Moderate 61–90, Poor 91–120, Very poor 121–250, Severe > 250 (µg/m³).

In [22]:
bands = [(0, 30, "Good"), (30, 60, "Satisfactory"), (60, 90, "Moderate"), (90, 120, "Poor"), (120, 250, "Very poor"), (250, 10_000, "Severe")]
d2 = daily[daily.pm25.notna()].copy()
d2["band"] = pd.cut(d2.pm25, [b[0] for b in bands] + [10_000], labels=[b[2] for b in bands])
d2["month_name"] = d2.month.map(MONTH_NAME)
share = (d2.groupby(["month_name", "band"], observed=False).size() / d2.groupby("month_name").size() * 100).unstack().reindex(list(MONTH_NAME.values()))
band_colors = ["#cde2fb", "#9ec5f4", "#5598e7", "#2a78d6", "#184f95", "#0d366b"]
fig11 = go.Figure()
for (lo, hi, name), col in zip(bands, band_colors):
    fig11.add_trace(go.Bar(y=share.index, x=share[name], name=name, orientation="h", marker=dict(color=col, line=dict(color="white", width=2)),
                           hovertemplate="%{y}<br>" + name + ": %{x:.0f}% of days<extra></extra>"))
fig11.update_layout(barmode="stack", bargap=0.35)
style(fig11, "In November, 98% of days were 'very poor' or 'severe'; in late September, none were", "Share of days, all three years (%)", "", height=330)
fig11.update_yaxes(autorange="reversed", showgrid=False); fig11.update_xaxes(range=[0, 100], showgrid=True, gridcolor=GRID)
save(fig11, "fig11_aqi_bands")
RESULTS["aqi_band_share"] = share.round(0).to_dict(orient="index")
share.round(0)

band,Good,Satisfactory,Moderate,Poor,Very poor,Severe
month_name,,,,,,
September (15-30),15.0,65.0,19.0,2.0,0.0,0.0
October,1.0,14.0,29.0,14.0,40.0,2.0
November,0.0,0.0,0.0,2.0,57.0,41.0


### Chart 12: the daily cycle (average PM2.5 by hour of day)

In [23]:
dc = city.assign(hour=city.index.hour, month=city.index.month).groupby(["month", "hour"]).pm25.mean().unstack(0)
fig12 = go.Figure()
for m in [9, 10, 11]:
    fig12.add_trace(go.Scatter(x=dc.index, y=dc[m], name=MONTH_NAME[m], mode="lines+markers", line=dict(color=MONTH_COLOR[m], width=2, dash=MONTH_DASH[m]),
                               marker=dict(size=5), hovertemplate="%{x}:00<br>%{y:.0f} µg/m³<extra>" + MONTH_NAME[m] + "</extra>"))
style(fig12, "Pollution builds through the night and clears in the afternoon; in November the night peak is about twice the afternoon low",
      "Hour of day (IST)", "Average city PM2.5 (µg/m³)")
fig12.update_xaxes(tickvals=list(range(0, 24, 3)), ticktext=[f"{h:02d}:00" for h in range(0, 24, 3)])
for m in [9, 10, 11]:
    fig12.add_annotation(x=23, y=dc[m].iloc[-1], text=MONTH_NAME[m].split(" ")[0], xanchor="left", xshift=8, showarrow=False,
                         font=dict(color=MONTH_COLOR[m] if m != 9 else INK2, size=12))
fig12.update_layout(showlegend=False, margin=dict(r=90))
save(fig12, "fig12_daily_cycle")
RESULTS["daily_cycle_nov"] = {"max_hour": int(dc[11].idxmax()), "max": round(dc[11].max()), "min_hour": int(dc[11].idxmin()), "min": round(dc[11].min())}
print(RESULTS["daily_cycle_nov"])

{'max_hour': 1, 'max': 296, 'min_hour': 16, 'min': 153}


### Chart 13: Diwali night 2025, station by station

In [24]:
D25 = DIWALI[2025]
w = d[(d.ts_ist >= D25 + pd.Timedelta(hours=12)) & (d.ts_ist <= D25 + pd.Timedelta(hours=36))]
hm = w.pivot_table(index="station_key", columns="ts_ist", values="pm25")
hm = hm.loc[hm.max(axis=1).sort_values().index]
fig13 = go.Figure(go.Heatmap(z=hm.values, x=hm.columns, y=[NAMES[k] for k in hm.index], colorscale=SEQ_SCALE, zmin=0, zmax=1500,
                             colorbar=dict(title="µg/m³", thickness=12), xgap=1, ygap=2,
                             hovertemplate="%{y}<br>%{x|%d %b %H:%M}<br>%{z:.0f} µg/m³<extra></extra>"))
style(fig13, "On Diwali night 2025 every station peaked between 1 am and 5 am", "Time (IST), noon on 20 Oct to noon on 21 Oct", "", height=430)
fig13.update_xaxes(tickformat="%d %b %H:%M"); fig13.update_yaxes(showgrid=False)
save(fig13, "fig13_diwali_station_heatmap")
peaks = w.loc[w.groupby("station_key").pm25.idxmax(), ["station_key", "ts_ist", "pm25"]]
RESULTS["diwali2025_station_peaks"] = {"earliest": peaks.ts_ist.min().strftime("%H:%M"), "latest": peaks.ts_ist.max().strftime("%d %b %H:%M"),
                                       "highest_station": NAMES[peaks.loc[peaks.pm25.idxmax(), "station_key"]], "highest": round(peaks.pm25.max())}
peaks.assign(name=peaks.station_key.map(NAMES)).sort_values("pm25", ascending=False)[["name", "ts_ist", "pm25"]].round(0)

,name,ts_ist,pm25
32285,Najafgarh,2025-10-21 05:00:00,1901.0
10109,Dwarka Sec-8,2025-10-21 05:00:00,1869.0
54459,Pusa,2025-10-21 03:00:00,1847.0
43369,Okhla Ph-2,2025-10-21 01:00:00,1806.0
60003,R K Puram,2025-10-21 03:00:00,1753.0
15651,ITO,2025-10-21 03:00:00,1556.0
4562,Anand Vihar,2025-10-21 02:00:00,1444.0
26741,Lodhi Road,2025-10-21 05:00:00,1444.0
48913,Punjabi Bagh,2025-10-21 01:00:00,1432.0
21193,Jahangirpuri,2025-10-21 01:00:00,970.0


### Chart 14: which way was the wind blowing?
We split October–November hours by the direction the wind came **from** (8 compass sectors) and average the city's PM2.5 in each. Punjab and Haryana, where most crop residue is burnt, lie to Delhi's north-west.

In [25]:
on = city[city.index.month >= 10].copy()
on = on.join(d.drop_duplicates("ts_ist").set_index("ts_ist")[["wind_dir_deg"]])
sectors = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
on["sector"] = pd.cut((on.wind_dir_deg + 22.5) % 360, np.arange(0, 361, 45), labels=sectors, right=False)
ws = on.groupby("sector", observed=False).agg(pm25=("pm25", "mean"), hours=("pm25", "size"))
ws["share"] = ws.hours / ws.hours.sum() * 100
fig14 = go.Figure(go.Barpolar(r=ws.pm25, theta=sectors, marker=dict(color=[HIGHLIGHT if s == "NW" else YEAR_COLOR[2024] for s in sectors],
                              line=dict(color="white", width=2)), customdata=ws[["share"]].values,
                              hovertemplate="Wind from %{theta}<br>%{r:.0f} µg/m³ average<br>%{customdata[0]:.0f}% of hours<extra></extra>"))
style(fig14, "West and north-west winds blow 60% of the time; west winds bring the dirtiest air", None, None, height=470)
fig14.update_traces(marker_color=[HIGHLIGHT if s_ == "W" else YEAR_COLOR[2024] for s_ in sectors])
fig14.update_layout(polar=dict(radialaxis=dict(ticksuffix=" ", title=dict(text="avg PM2.5 (µg/m³)", font=dict(size=11)), gridcolor=GRID, angle=90, tickfont=dict(size=10)),
                               angularaxis=dict(direction="clockwise", rotation=90, gridcolor=GRID), bgcolor="white"))
save(fig14, "fig14_wind_direction")
RESULTS["wind_sectors"] = ws.round(0).to_dict(orient="index")
ws.round(0)

,pm25,hours,share
sector,,,
N,165.0,378,9.0
NE,183.0,266,6.0
E,187.0,333,8.0
SE,174.0,362,8.0
S,175.0,233,5.0
SW,164.0,215,5.0
W,206.0,1309,30.0
NW,157.0,1296,30.0


### Chart 15: the 11 stations on a map

In [26]:
locs_json = json.loads((ROOT / "raw" / "openaq" / "openaq_locations_delhi_bbox.json").read_text(encoding="utf-8"))
coords = {r["id"]: (r["coordinates"]["latitude"], r["coordinates"]["longitude"]) for r in locs_json["results"]}
mt = pd.read_csv(INTERIM / "station_match_cpcb_openaq.csv").set_index("station_key")["openaq_id"]
geo = pd.DataFrame({k: coords[int(mt[k if k != "pusa" else "pusa_dpcc"])] for k in NAMES}, index=["lat", "lon"]).T
geo["octnov"] = d[d.ts_ist.dt.month >= 10].groupby("station_key").pm25.mean()
geo["name"] = geo.index.map(NAMES)
fig15 = go.Figure(go.Scattermap(lat=geo.lat, lon=geo.lon, mode="markers+text", text=geo.name, textposition="top right",
                                textfont=dict(size=12, color=INK),
                                marker=dict(size=(geo.octnov - geo.octnov.min()) / (geo.octnov.max() - geo.octnov.min()) * 22 + 12,
                                            color=geo.octnov, colorscale=[[i / (len(SEQ) - 3), c] for i, c in enumerate(SEQ[2:])], cmin=geo.octnov.min(), cmax=geo.octnov.max(),
                                            colorbar=dict(title="µg/m³", thickness=12)),
                                customdata=geo[["octnov"]].values, hovertemplate="%{text}<br>Oct-Nov average %{customdata[0]:.0f} µg/m³<extra></extra>"))
fig15.update_layout(map=dict(style="carto-positron", center=dict(lat=28.64, lon=77.18), zoom=9.6), height=520,
                    margin=dict(l=0, r=0, t=60, b=0), title=dict(text="Jahangirpuri (north) and Anand Vihar (east) are the most polluted stations; Lodhi Road the least",
                    x=0, xanchor="left", font=dict(size=16, color=INK)), paper_bgcolor="white",
                    font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color=INK2))
save(fig15, "fig15_station_map")
geo.sort_values("octnov", ascending=False).round(1)

,lat,lon,octnov,name
jahangirpuri,28.7,77.2,216.0,Jahangirpuri
anand_vihar,28.6,77.3,212.8,Anand Vihar
punjabi_bagh,28.7,77.1,200.8,Punjabi Bagh
rk_puram,28.6,77.2,190.1,R K Puram
narela,28.8,77.1,187.6,Narela
dwarka_sec8,28.6,77.1,183.6,Dwarka Sec-8
okhla_ph2,28.5,77.3,172.2,Okhla Ph-2
ito,28.6,77.2,168.8,ITO
najafgarh,28.6,76.9,154.4,Najafgarh
pusa,28.6,77.1,153.8,Pusa


### Chart 16: station ranking, every year

In [27]:
rank = d[d.ts_ist.dt.month >= 10].groupby(["station_key", "year"]).pm25.mean().unstack()
order_r = rank.mean(axis=1).sort_values().index
fig16 = go.Figure()
for y in YEARS:
    fig16.add_trace(go.Scatter(x=rank.loc[order_r, y], y=[NAMES[k] for k in order_r], mode="markers", name=str(y),
                               marker=dict(color=YEAR_COLOR[y], size=11, symbol={2023: "circle-open", 2024: "diamond", 2025: "circle"}[y],
                                           line=dict(width=2, color=YEAR_COLOR[y])),
                               hovertemplate="%{y}<br>%{x:.0f} µg/m³<extra>" + str(y) + "</extra>"))
style(fig16, "The ranking is stable: the same top three and the same cleanest station every year", "Oct-Nov average PM2.5 (µg/m³)", "", height=470)
fig16.update_xaxes(showgrid=True, gridcolor=GRID); fig16.update_yaxes(showgrid=False)
save(fig16, "fig16_station_ranking")
RESULTS["station_octnov_mean"] = rank.mean(axis=1).round(0).rename(index=NAMES).sort_values(ascending=False).to_dict()
rank.round(0).loc[order_r[::-1]]

year,2023,2024,2025
station_key,,,
jahangirpuri,216.0,222.0,210.0
anand_vihar,209.0,232.0,198.0
punjabi_bagh,207.0,206.0,189.0
rk_puram,195.0,190.0,186.0
narela,183.0,192.0,188.0
dwarka_sec8,188.0,189.0,173.0
okhla_ph2,179.0,175.0,163.0
ito,168.0,163.0,176.0
najafgarh,158.0,164.0,141.0


### Chart 17: how well does the weather model fit?

In [28]:
fd = daily[daily.pm25.notna()]
dw_mask = fd.dd == 0
fig17 = go.Figure()
lim = [0, max(fd.pm25.max(), fd.pred_full.max()) * 1.05]
fig17.add_trace(go.Scatter(x=lim, y=lim, mode="lines", line=dict(color=INK2, width=1, dash="dot"), name="Perfect prediction", hoverinfo="skip"))
fig17.add_trace(go.Scatter(x=fd.loc[~dw_mask, "pred_full"], y=fd.loc[~dw_mask, "pm25"], mode="markers", name="Other days",
                           marker=dict(color=CONTEXT, size=7, line=dict(color="white", width=1)), customdata=fd.index[~dw_mask].strftime("%d %b %Y"),
                           hovertemplate="%{customdata}<br>predicted %{x:.0f}, actual %{y:.0f}<extra></extra>"))
fig17.add_trace(go.Scatter(x=fd.loc[dw_mask, "pred_full"], y=fd.loc[dw_mask, "pm25"], mode="markers+text", name="Diwali day",
                           marker=dict(color=HIGHLIGHT, size=12, line=dict(color="white", width=2)), text=[f"Diwali {y}" for y in fd.loc[dw_mask, "year"]],
                           textposition="middle right", textfont=dict(color=INK, size=11),
                           hovertemplate="predicted %{x:.0f}, actual %{y:.0f}<extra></extra>"))
style(fig17, f"The full-weather model follows the season (R² = {fits['full'].rsquared:.2f}) but misses the sharpest spikes",
      "Predicted from weather (µg/m³)", "Actual daily city PM2.5 (µg/m³)")
fig17.update_xaxes(range=lim); fig17.update_yaxes(range=lim)
save(fig17, "fig17_model_fit")

### Chart 18: is the modelled weather right? (Open-Meteo vs the stations' own thermometers)

In [29]:
stt = d.groupby("ts_ist").st_temp_c.median()
tcmp = pd.concat([d.drop_duplicates("ts_ist").set_index("ts_ist").temp_c, stt], axis=1, keys=["model", "stations"]).dropna()
r_t = tcmp.corr().iloc[0, 1]
fig18 = go.Figure(go.Histogram2d(x=tcmp.model, y=tcmp.stations, nbinsx=40, nbinsy=40, zmin=0.5,
                                 colorscale=[[0, "#ffffff"], [0.001, SEQ[0]]] + [[(i + 1) / len(SEQ), c] for i, c in enumerate(SEQ)][1:], colorbar=dict(title="hours", thickness=12),
                                 hovertemplate="model %{x:.0f} °C, stations %{y:.0f} °C<br>%{z} hours<extra></extra>"))
lt = [tcmp.min().min(), tcmp.max().max()]
fig18.add_trace(go.Scatter(x=lt, y=lt, mode="lines", line=dict(color=HIGHLIGHT, width=2, dash="dash"), name="Equal", hoverinfo="skip"))
style(fig18, f"Modelled and measured temperature agree closely (r = {r_t:.2f}); stations run about {(tcmp.stations - tcmp.model).mean():.1f} °C warmer",
      "Open-Meteo temperature (°C)", "Median of station thermometers (°C)")
save(fig18, "fig18_weather_check")
RESULTS["temp_check"] = {"r": round(r_t, 2), "station_minus_model": round((tcmp.stations - tcmp.model).mean(), 1)}

### Chart 19: where were the gaps in the raw data?

In [30]:
gap = d.assign(day=d.ts_ist.dt.normalize(), miss=d.pm25_raw.isna()).groupby(["station_key", "day"]).miss.sum().unstack()
gap.index = [NAMES[k] for k in gap.index]
gap = gap.loc[gap.sum(axis=1).sort_values().index]
fig19 = go.Figure(go.Heatmap(z=gap.values, x=list(range(gap.shape[1])), y=gap.index, colorscale=[[0, "#f4f6f8"], [0.0001, SEQ[1]], [1, SEQ[-1]]],
                             zmin=0, zmax=24, colorbar=dict(title="hours<br>missing", thickness=12), xgap=0.5, ygap=2,
                             customdata=np.tile([c.strftime("%d %b %Y") for c in gap.columns], (gap.shape[0], 1)),
                             hovertemplate="%{y}<br>%{customdata}<br>%{z} of 24 hours missing<extra></extra>"))
ticks = [i for i, c in enumerate(gap.columns) if c.day == 15 or (c.month == 9 and c.day == 15)]
style(fig19, "Most gaps last a few hours; only Lodhi Road lost many whole days (11)", "Day (three seasons placed side by side)", "", height=430)
fig19.update_xaxes(tickvals=ticks, ticktext=[gap.columns[i].strftime("%d %b %y") for i in ticks], showgrid=False); fig19.update_yaxes(showgrid=False)
for i in range(1, 3):
    fig19.add_vline(x=i * 77 - 0.5, line=dict(color=INK, width=1.5))
save(fig19, "fig19_missing_heatmap")
RESULTS["station_days_fully_missing"] = int((gap == 24).sum().sum())

### Chart 20: where did the errors come from?

In [31]:
err = d[d.pm25_flag.isin(["impossible_value", "stuck", "isolated_spike"])].groupby(["station_key", "pm25_flag"], observed=True).size().unstack(fill_value=0)
err = err.reindex(list(NAMES), fill_value=0); err.index = [NAMES[k] for k in err.index]
err["total"] = err.sum(axis=1); err = err.sort_values("total")
labels = {"impossible_value": "Impossible (≤ 0)", "stuck": "Stuck sensor", "isolated_spike": "Isolated spike"}
fig20 = go.Figure()
for col, c in zip(["impossible_value", "stuck", "isolated_spike"], ["#9ec5f4", "#2a78d6", "#0d366b"]):
    if col in err:
        fig20.add_trace(go.Bar(y=err.index, x=err[col], name=labels[col], orientation="h", marker=dict(color=c, line=dict(color="white", width=2)),
                               hovertemplate="%{y}<br>" + labels[col] + ": %{x} hours<extra></extra>"))
fig20.update_layout(barmode="stack", bargap=0.3)
style(fig20, f"{int(err.total.sum())} error hours were removed; Lodhi Road ({int(err.loc['Lodhi Road','total'])}) and ITO ({int(err.loc['ITO','total'])}) account for over half", "Hours removed", "", height=430)
fig20.update_yaxes(showgrid=False); fig20.update_xaxes(showgrid=True, gridcolor=GRID)
save(fig20, "fig20_errors_by_station")
err

pm25_flag,impossible_value,isolated_spike,stuck,total
Okhla Ph-2,0,0,0,0
Jahangirpuri,0,0,0,0
Narela,0,0,0,0
R K Puram,0,4,0,4
Najafgarh,0,3,4,7
Dwarka Sec-8,0,0,8,8
Pusa,8,4,0,12
Anand Vihar,0,13,0,13
Punjabi Bagh,0,0,14,14
ITO,28,2,0,30


## 6b · Charts inspired by the course notes and the intro lecture

### Chart 21: every hour of every season (heatmap, hour of day × date)
A heatmap shows where in a table the high values are (lecture 7). Each column is a day, each row an hour; darker is dirtier.

In [32]:
fig21 = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06, subplot_titles=[str(y) for y in YEARS])
for i, y in enumerate(YEARS, start=1):
    sub = city[city.year == y]
    hm2 = sub.assign(day=pd.to_datetime("2000-" + sub.index.strftime("%m-%d")), hour=sub.index.hour).pivot_table(index="hour", columns="day", values="pm25")
    fig21.add_trace(go.Heatmap(z=hm2.values, x=hm2.columns, y=hm2.index, colorscale=SEQ_SCALE, zmin=0, zmax=600, showscale=(i == 1),
                               colorbar=dict(title="µg/m³", thickness=12, len=0.9),
                               customdata=np.tile([c.strftime("%d %b") for c in hm2.columns], (24, 1)),
                               hovertemplate=f"%{{customdata}} {y}, %{{y}}:00<br>%{{z:.0f}} µg/m³<extra></extra>"), row=i, col=1)
    dwt = pd.Timestamp(2000, DIWALI[y].month, DIWALI[y].day)
    fig21.add_trace(go.Scatter(x=[dwt], y=[-1.6], mode="markers", marker=dict(symbol="triangle-up", size=11, color=HIGHLIGHT),
                               showlegend=False, hovertemplate=f"Diwali {y}<extra></extra>"), row=i, col=1)
    fig21.update_yaxes(tickvals=[0, 6, 12, 18], ticktext=["00:00", "06:00", "12:00", "18:00"], range=[-2.5, 23.5], row=i, col=1)
style(fig21, "Every night of the season is a dark band; Diwali (orange triangle) is the darkest single column in 2025", None, None, height=640)
fig21.update_xaxes(tickformat="%d %b", showgrid=False); fig21.update_yaxes(showgrid=False)
for a_ in fig21.layout.annotations:
    a_.font = dict(size=13, color=INK); a_.x = 0; a_.xanchor = "left"
save(fig21, "fig21_hour_day_heatmap")

### Chart 22: a 24-hour rose, after Florence Nightingale (1858)
Nightingale's diagram drew each month as a wedge whose **area** shows the deaths. Here each hour of the day is a wedge whose area shows the average city PM2.5. To keep area honest, the radius is the square root of the value (lecture 7: plotly scales bubbles by area for the same reason).

In [33]:
fig22 = make_subplots(rows=1, cols=2, specs=[[{"type": "polar"}, {"type": "polar"}]], subplot_titles=["Late September", "November"])
rmax = np.sqrt(dc.max().max()) * 1.05
for j, m in enumerate([9, 11], start=1):
    vals = dc[m]
    colors = [HIGHLIGHT if (m == 11 and h == vals.idxmax()) else MONTH_COLOR[m] for h in vals.index]
    fig22.add_trace(go.Barpolar(r=np.sqrt(vals.values), theta=vals.index * 15, width=[14] * 24,
                                marker=dict(color=colors, line=dict(color="white", width=1)), showlegend=False,
                                customdata=np.stack([vals.index, vals.values], axis=1),
                                hovertemplate="%{customdata[0]}:00 - average %{customdata[1]:.0f} µg/m³<extra></extra>"), row=1, col=j)
pol = dict(radialaxis=dict(range=[0, rmax], showticklabels=False, showline=False, ticks="", gridcolor=GRID), bgcolor="white",
           angularaxis=dict(direction="clockwise", rotation=90, tickvals=[0, 90, 180, 270], ticktext=["midnight", "6 am", "noon", "6 pm"], gridcolor=GRID))
fig22.update_layout(polar=pol, polar2=pol)
style(fig22, f"November's air is dirtiest around {dc[11].idxmax()} am ({dc[11].max():.0f} µg/m³) and cleanest around {dc[11].idxmin() - 12} pm; late September is small at every hour",
      None, None, height=470)
for a_ in fig22.layout.annotations:
    a_.font = dict(size=14, color=INK); a_.y = a_.y + 0.11
fig22.update_layout(margin=dict(t=120))
save(fig22, "fig22_nightingale_rose")

### Chart 23: how the spread of daily PM2.5 moves, week by week (ridgeline)

In [34]:
dd3 = daily[daily.pm25.notna()].copy()
dd3["week"] = ((dd3.index.to_series() - pd.to_datetime(dd3.year.astype(str) + "-09-15")).dt.days // 7).clip(upper=10)
wk_start = pd.date_range("2000-09-15", periods=11, freq="7D")
labels_w = [f"{t:%d %b}" for t in wk_start]
fig23 = go.Figure()
ramp = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#104281"]
for w_ in range(11):
    v = dd3[dd3.week == w_].pm25
    fig23.add_trace(go.Violin(x=v, y=[labels_w[w_]] * len(v), orientation="h", side="positive", width=2.4, points=False,
                              line=dict(color="white", width=1.5), fillcolor=ramp[w_], opacity=1,
                              meanline=dict(visible=False), showlegend=False, spanmode="hard", scalemode="width",
                              hoverinfo="skip"))
    fig23.add_trace(go.Scatter(x=[v.median()], y=[labels_w[w_]], mode="markers", marker=dict(symbol="line-ns", size=14, line=dict(color=HIGHLIGHT, width=3)),
                               showlegend=False, hovertemplate=f"Week from {labels_w[w_]}: median %{{x:.0f}} µg/m³ ({len(v)} days)<extra></extra>"))
style(fig23, "Week by week the whole distribution slides right and spreads out; orange ticks mark each week's median", "Daily city-wide PM2.5 (µg/m³)", "Week starting", height=560)
fig23.update_yaxes(categoryorder="array", categoryarray=labels_w[::-1], showgrid=False); fig23.update_xaxes(range=[0, 650], showgrid=True, gridcolor=GRID)
RESULTS["weekly_median_first_last"] = [round(dd3[dd3.week == 0].pm25.median()), round(dd3[dd3.week == 9].pm25.median())]
save(fig23, "fig23_ridgeline")

### Chart 24: each station from late September to November (slope chart)
Position on a common scale is read most accurately (lecture 9), so each station is two dots joined by a line.

In [35]:
sp = d.assign(m=d.ts_ist.dt.month).groupby(["station_key", "m"]).pm25.mean().unstack()[[9, 11]]
sp["rise"] = sp[11] - sp[9]
top = sp.rise.idxmax()
fig24 = go.Figure()
for k, r_ in sp.sort_values(11).iterrows():
    hl = k == top
    fig24.add_trace(go.Scatter(x=["Late September", "November"], y=[r_[9], r_[11]], mode="lines+markers",
                               line=dict(color=HIGHLIGHT if hl else CONTEXT, width=3 if hl else 1.5),
                               marker=dict(size=8 if hl else 6, color=HIGHLIGHT if hl else CONTEXT), showlegend=False,
                               hovertemplate=f"{NAMES[k]}<br>%{{x}}: %{{y:.0f}} µg/m³<extra></extra>"))
# labels on the right, nudged apart so they do not overlap
lab = sp.sort_values(11)[11].copy(); ypos = []
for v in lab.values:
    ypos.append(max(v, ypos[-1] + 9) if ypos else v)
for (k, v), yp in zip(lab.items(), ypos):
    fig24.add_annotation(x=1, y=yp, text=f"{NAMES[k]} {v:.0f}", xanchor="left", xshift=10, showarrow=False,
                         font=dict(size=11, color="#c4501f" if k == top else INK2))
style(fig24, f"Every station climbs steeply; {NAMES[top]} rises the most (+{sp.rise.max():.0f} µg/m³)", "", "Average PM2.5 (µg/m³)", height=560)
fig24.update_layout(margin=dict(r=150)); fig24.update_xaxes(range=[-0.15, 1.05])
RESULTS["slope"] = {"top": NAMES[top], "rise": round(sp.rise.max()), "min_rise": round(sp.rise.min()), "min_station": NAMES[sp.rise.idxmin()],
                    "sep_range": [round(sp[9].min()), round(sp[9].max())], "nov_range": [round(sp[11].min()), round(sp[11].max())]}
save(fig24, "fig24_slope")

### Chart 25: Diwali night 2025 on the map, hour by hour (after John Snow and Hans Rosling)
Snow put each death where it happened; Rosling set the bubbles moving through time. Press play to watch the night.

In [36]:
hrs25 = pd.date_range(DIWALI[2025] + pd.Timedelta(hours=16), DIWALI[2025] + pd.Timedelta(hours=34), freq="h")
mv = d[d.ts_ist.isin(hrs25)].pivot_table(index="ts_ist", columns="station_key", values="pm25")
def frame_trace(t):
    v = mv.loc[t, geo.index]
    return go.Scattermap(lat=geo.lat, lon=geo.lon, mode="markers", text=geo.name,
                         marker=dict(size=np.clip(np.sqrt(v.fillna(0)) * 1.4, 6, 70), color=v, colorscale=SEQ_SCALE, cmin=0, cmax=1500,
                                     colorbar=dict(title="µg/m³", thickness=12)),
                         customdata=np.stack([v.round(0).values], axis=1), hovertemplate="%{text}<br>%{customdata[0]:.0f} µg/m³<extra></extra>")
fig25 = go.Figure(data=[frame_trace(hrs25[0])], frames=[go.Frame(data=[frame_trace(t)], name=t.strftime("%d %b %H:%M")) for t in hrs25])
steps = [dict(method="animate", label=t.strftime("%d %b %H:%M"), args=[[t.strftime("%d %b %H:%M")], dict(mode="immediate", frame=dict(duration=600, redraw=True), transition=dict(duration=0))]) for t in hrs25]
fig25.update_layout(map=dict(style="carto-positron", center=dict(lat=28.64, lon=77.16), zoom=9.4), height=560,
                    margin=dict(l=0, r=0, t=60, b=0), paper_bgcolor="white",
                    title=dict(text="Diwali night 2025: smoke spreads across the city from 9 pm and peaks between 1 and 5 am", x=0, xanchor="left", font=dict(size=16, color=INK)),
                    font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color=INK2),
                    sliders=[dict(active=0, steps=steps, x=0.08, len=0.9, y=0.02, currentvalue=dict(prefix="", font=dict(size=13, color=INK)))],
                    updatemenus=[dict(type="buttons", x=0.0, y=0.02, xanchor="left", yanchor="bottom", showactive=False,
                                      buttons=[dict(label="▶ Play", method="animate", args=[None, dict(frame=dict(duration=600, redraw=True), fromcurrent=True)])])])
save(fig25, "fig25_diwali_map_animation")

### A look at the data itself
The first rows of the clean table, shown on the website before the charts.

In [37]:
preview_cols = ["station_key", "ts_ist", "pm25", "pm25_source", "temp_c", "wind_kmh", "blh_m"]
pv = d[(d.ts_ist >= DIWALI[2025] + pd.Timedelta(hours=22)) & (d.station_key.isin(["anand_vihar", "ito", "rk_puram"]))].sort_values(["ts_ist", "station_key"]).head(6)[preview_cols].copy()
pv["station_key"] = pv.station_key.map(NAMES); pv["ts_ist"] = pv.ts_ist.dt.strftime("%Y-%m-%d %H:%M")
pv = pv.round(1)
PREVIEW = {"columns": ["station", "time (IST)", "PM2.5 µg/m³", "source", "temp °C", "wind km/h", "mixing height m"], "rows": pv.values.tolist(),
           "shape": [int(len(d)), int(d.shape[1])]}
pv

,station_key,ts_ist,pm25,pm25_source,temp_c,wind_kmh,blh_m
4558,Anand Vihar,2025-10-20 22:00,627.0,cpcb,26.5,3.2,155.0
15646,ITO,2025-10-20 22:00,469.5,cpcb,26.5,3.2,155.0
59998,R K Puram,2025-10-20 22:00,832.0,cpcb,26.5,3.2,155.0
4559,Anand Vihar,2025-10-20 23:00,1012.0,openaq,25.6,3.9,120.0
15647,ITO,2025-10-20 23:00,614.0,cpcb,25.6,3.9,120.0
59999,R K Puram,2025-10-20 23:00,1267.0,openaq,25.6,3.9,120.0


## 7 · Save the numbers the website and report use

In [38]:
def clean_keys(o):
    if isinstance(o, dict):
        return {str(k): clean_keys(v) for k, v in o.items()}
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return o
(DOCS / "assets" / "results.json").write_text(json.dumps(clean_keys(RESULTS), indent=1), encoding="utf-8")
print(json.dumps(clean_keys(RESULTS), indent=1))

{
 "sep_mean": {
  "2023": 47.0,
  "2024": 51.9,
  "2025": 47.5
 },
 "octnov_mean": {
  "2023": 179.9,
  "2024": 183.6,
  "2025": 173.8
 },
 "rise": {
  "2023": 132.9,
  "2024": 131.7,
  "2025": 126.3
 },
 "pct_days_over_naaqs_octnov": {
  "2023": 95.0,
  "2024": 93.0,
  "2025": 89.0
 },
 "daily_median_by_month": {
  "2023": {
   "9": 48.0,
   "10": 92.0,
   "11": 260.0
  },
  "2024": {
   "9": 49.0,
   "10": 110.0,
   "11": 226.0
  },
  "2025": {
   "9": 45.0,
   "10": 104.0,
   "11": 229.0
  }
 },
 "spearman_daily": {
  "blh": -0.21,
  "wind": -0.36,
  "temp": -0.74,
  "rh": -0.3,
  "rain": -0.46
 },
 "model_r2": {
  "ventilation": 0.2,
  "full": 0.63
 },
 "model_r2_unseen_year": {
  "full": {
   "2023": 0.63,
   "2024": 0.48,
   "2025": 0.44
  },
  "ventilation": {
   "2023": 0.15,
   "2024": 0.18,
   "2025": -0.23
  }
 },
 "weather_rise": {
  "2023": {
   "rise_ventilation": 4.6,
   "rise_full": 96.7
  },
  "2024": {
   "rise_ventilation": 22.9,
   "rise_full": 95.0
  },
  "2025": 

## 8 · Bundle the charts for the website
The website in `docs/` must also open offline (double-clicking `index.html`), where browsers block loading separate JSON files. So we pack all figures into one JavaScript file, and save the matching Plotly library next to it.

In [39]:
from plotly.offline import get_plotlyjs
figs = {p.stem: json.loads(p.read_text(encoding="utf-8")) for p in sorted(FIG_DIR.glob("fig*.json"))}
(DOCS / "assets" / "figures.js").write_text("window.FIGS = " + json.dumps(figs) + ";\nwindow.PREVIEW = " + json.dumps(PREVIEW, default=str) + ";", encoding="utf-8")
(DOCS / "assets" / "vendor").mkdir(exist_ok=True)
(DOCS / "assets" / "vendor" / "plotly.min.js").write_text(get_plotlyjs(), encoding="utf-8")
print("Bundled", len(figs), "figures for the website")

Bundled 25 figures for the website
